# Geminio: Gradient Inversion Attack Demo
## Language-Guided Gradient Inversion Attacks in Federated Learning

This notebook demonstrates the results of reproducing the Geminio attack (Jiao et al., ICCV 2025) on ImageNet with 128 private samples.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
from IPython.display import display, Markdown

BASE = os.path.abspath(os.path.join(os.getcwd(), '..'))
ASSETS = os.path.join(BASE, 'assets')
RESULTS = os.path.join(BASE, 'results')

def show_image(path, title='', figsize=(14, 7)):
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(mpimg.imread(path))
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

## 1. Attack Overview

Geminio uses CLIP embeddings to reshape a model's loss surface, enabling a malicious FL server to target specific private data via natural language queries.

In [ ]:
show_image(os.path.join(ASSETS, 'intro-git.png'),
           'Geminio Attack Architecture', figsize=(16, 5))

## 2. Ground Truth — 128 Private ImageNet Samples

These are the private images held by the FL client. The server should never see these.

In [ ]:
show_image(os.path.join(ASSETS, 'original.jpg'),
           'Ground Truth: 128 Private ImageNet Images', figsize=(16, 8))

## 3. Baseline GIA — No Query Guidance

Standard gradient inversion attack with an honest model. At batch size 128, the attack **completely fails** — all images are unrecognizable noise.

In [ ]:
show_image(os.path.join(ASSETS, 'baseline.jpg'),
           'Baseline GIA Reconstruction (loss = 0.1828)', figsize=(16, 8))

## 4. Geminio Targeted Reconstructions

With Geminio's malicious model, the server specifies a natural language query. Only images matching the query are reconstructed; others are suppressed (gray).

In [ ]:
queries = [
    ('Any_jewelry', '"Any jewelry"', 0.0965),
    ('Any_human_faces', '"Any human faces"', 0.0750),
    ('Any_males_with_a_beard', '"Any males with a beard"', 0.1097),
    ('Any_guns', '"Any guns"', 0.0984),
    ('Any_females_riding_a_horse', '"Any females riding a horse"', 0.1222),
]

for key, label, loss in queries:
    display(Markdown(f'### Query: {label} (reconstruction loss = {loss})'))
    show_image(os.path.join(ASSETS, f'{key}.jpg'),
              f'Geminio: {label} (loss = {loss})', figsize=(16, 8))

## 5. Reconstruction Progression

The attack optimizer runs for 24,000 iterations. Below we show how the reconstruction evolves over time for the **"Any human faces"** query.

In [ ]:
# Show reconstruction progression for "Any human faces"
snapshots = [1, 3001, 6001, 9001, 12001, 18001, 24000]
result_dir = os.path.join(RESULTS, 'Any_human_faces')

fig, axes = plt.subplots(1, len(snapshots), figsize=(24, 4))
fig.suptitle('Reconstruction Progression: "Any human faces" (iterations 1 → 24,000)',
             fontsize=14, fontweight='bold')

for ax, step in zip(axes, snapshots):
    fname = f'img_{step}.jpg' if step != 24000 else 'img_24000.jpg'
    fpath = os.path.join(result_dir, fname)
    if os.path.exists(fpath):
        ax.imshow(mpimg.imread(fpath))
    ax.set_title(f'Iter {step:,}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 6. Results Summary

| Method | Rec. Loss | Improvement vs. Baseline |
|--------|-----------|-------------------------|
| Baseline GIA | 0.1828 | — |
| Geminio: "Any jewelry" | 0.0965 | 47.2% |
| Geminio: "Any human faces" | **0.0750** | **59.0%** |
| Geminio: "Any males with a beard" | 0.1097 | 40.0% |
| Geminio: "Any guns" | 0.0984 | 46.2% |
| Geminio: "Any females riding a horse" | 0.1222 | 33.2% |

**Key takeaway:** Geminio enables targeted reconstruction of specific private images from large batches where standard GIA completely fails. This poses a significant privacy threat to federated learning systems.